# Aegis GRPO Training Notebook

This notebook reruns the Hugging Face TRL training path used in the repo and writes a fresh evidence bundle with loss and reward plots.

It is designed to work both in a local Jupyter session and in Colab after the repository has been mounted or cloned into the runtime.

## Runtime Notes

- If you have a GPU, the notebook will use `Qwen/Qwen3-0.6B`.
- On CPU-only runtimes it falls back to `sshleifer/tiny-gpt2`, which now works with the repo's tool-calling GRPO path.
- The notebook writes outputs to `reports/training_colab/` and model checkpoints to `artifacts/grpo-colab/`.

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path.cwd()
if not (REPO_DIR / 'pyproject.toml').exists():
    raise RuntimeError('Run this notebook from the aegis repo root or clone/mount the repo before continuing.')

os.chdir(REPO_DIR)
print(f'Repo root: {REPO_DIR}')

In [ ]:
%pip install -q -e .[training]

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'training.train', '--check-stack'], check=True)

In [ ]:
import shutil
import subprocess
import sys
import torch

model_name = 'Qwen/Qwen3-0.6B' if torch.cuda.is_available() else 'sshleifer/tiny-gpt2'
output_dir = 'artifacts/grpo-colab'
evidence_dir = 'reports/training_colab'

shutil.rmtree(output_dir, ignore_errors=True)
shutil.rmtree(evidence_dir, ignore_errors=True)

cmd = [
    sys.executable, '-m', 'training.train', '--train',
    '--model-name', model_name,
    '--episodes-per-attack', '1',
    '--max-steps', '3',
    '--per-device-train-batch-size', '2',
    '--gradient-accumulation-steps', '1',
    '--num-generations', '2',
    '--max-completion-length', '48',
    '--max-tool-calling-iterations', '4',
    '--logging-steps', '1',
    '--save-steps', '10',
    '--output-dir', output_dir,
    '--evidence-dir', evidence_dir,
    '--run-name', 'aegis-grpo-colab',
]

result = subprocess.run(cmd, check=True, text=True, capture_output=True)
print(result.stdout)
print(f'Finished training with model: {model_name}')

In [ ]:
import json
from pathlib import Path

summary_path = Path('reports/training_colab/training_summary.json')
history_path = Path('reports/training_colab/training_log_history.json')

summary = json.loads(summary_path.read_text(encoding='utf-8'))
history = json.loads(history_path.read_text(encoding='utf-8'))
summary, history

In [ ]:
from IPython.display import Image, display

display(Image(filename='reports/training_colab/training_curves.png'))